# LocPop: reproducible community benchmark

This is the corrected NUMBA benchmark.  Shared experiment logic lives in
`BenchmarkNumbaExperiments.py`, which prevents the LocPop and LocStab protocols
from drifting apart.

The run uses fixed data seeds, ten paired permutations, standardized vector
features, disjoint friendship/enmity relations, correctly named adjusted Rand
index (ARI), same-run warm starts, convergence diagnostics, and explicit
algorithm parameters.  It writes both raw and summarized CSVs and regenerates
individual plus four-panel summary plots.

For community detection, the singleton initialization is omitted on Cora for
tractability and the predicted-$k$ initialization is omitted on Jazz because
that dataset has no reference $k$.  A predicted-$k$ Cora start has capacity 50,
while a Leiden start retains any larger capacity required by its initial labels.
All other datasets allow up to $n$ labels and therefore include singleton-
creation moves whenever an empty label is available.  Random-25 uses 25
connected ten-node Erdős--Rényi blocks ($p=0.40$), independent cross-block
edges with probability $q=0.001$, and one random bridge between consecutive
blocks on a ring, ensuring a connected but community-structured graph.

Expected outputs:

- `csv/PopularCommunity/runs.csv`
- `csv/PopularCommunity/results.csv`
- `csv/PopularCommunity/preprocessing.csv`
- `csv/PopularCommunity/data-diagnostics.csv`
- `csv/PopularCommunity/dataset-0.csv`, `dataset-1.csv`, `dataset-2.csv`
- `figures/PopularCommunity/*.png`


In [1]:
from importlib.metadata import version
from BenchmarkNumbaExperiments import run_community_experiment, DATA_SEED, THRESHOLDS, DOMAINS, RANDOM25_WITHIN_P, RANDOM25_BETWEEN_P

REPETITIONS = 10
LOCAL_STABLE = False

print("Data seed:", DATA_SEED)
print("Repetitions:", REPETITIONS)
print("Thresholds:", THRESHOLDS)
print("Domains:", DOMAINS)

print("Random-25 probabilities (within, between):", RANDOM25_WITHIN_P, RANDOM25_BETWEEN_P)
for package in ["numpy", "scipy", "pandas", "scikit-learn", "networkx", "numba", "matplotlib"]:
    print(f"{package}={version(package)}")


Data seed: 20260817
Repetitions: 10
Thresholds: ((0.2, 0.2), (0.25, 0.35), (0.4, 0.4))
Domains: ('B', 'AF', 'AE')
Random-25 probabilities (within, between): 0.4 0.001
numpy=2.5.2
scipy=1.18.0
pandas=3.0.5
scikit-learn=1.9.0
networkx=3.6.1
numba=0.67.0
matplotlib=3.11.1


## Execute the complete experiment

This cell performs the full production run and overwrites the corresponding
CSV and figure artifacts.  In particular, Cora's all-pairs preprocessing and
large-instance heuristic runs can take several minutes.  Do not interrupt the
kernel while files are being written.


In [2]:
summary = run_community_experiment(local_stable=LOCAL_STABLE, repetitions=REPETITIONS)
summary

,Method,Dataset,Preference,Initialization,Beta Friend,Beta Enemy,Repetitions,Adjusted Rand Index,Adjusted Rand Index SD,Modularity,...,Seconds,Seconds SD,Moves,Moves SD,Converged,Converged SD,Final Coalitions,Final Coalitions SD,Initial Adjusted Rand Index,Initial Adjusted Rand Index SD
0,Louvain,Karate Club,-,-,NaN,NaN,10,0.438855,0.034855,0.439534,...,0.006405,0.000936,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Leiden,Karate Club,-,-,NaN,NaN,10,0.426886,0.037952,0.440923,...,0.012331,0.001202,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,LocPop,Karate Club,B,Ld,0.2,0.2,10,0.132099,0.002195,0.294619,...,0.098610,0.292957,14.2,0.400000,1.0,0.0,16.2,0.600000,0.373084,0.018940
3,LocPop,Karate Club,B,S,0.2,0.2,10,0.096005,0.008734,0.250469,...,0.000863,0.000157,17.6,1.496663,1.0,0.0,18.8,0.400000,0.000000,0.000000
4,LocPop,Karate Club,B,P,0.2,0.2,10,0.111214,0.011134,0.267546,...,0.000835,0.000223,27.8,1.833030,1.0,0.0,18.4,0.800000,0.010145,0.022421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,LocPop,Random-25,AF,S,0.4,0.4,10,0.198753,0.010494,0.607812,...,0.039846,0.002070,319.9,20.097015,1.0,0.0,12.0,0.894427,0.000000,0.000000
94,LocPop,Random-25,AF,P,0.4,0.4,10,0.210303,0.013930,0.618450,...,0.024318,0.001629,262.6,14.623269,1.0,0.0,12.0,1.183216,0.000365,0.003473
95,LocPop,Random-25,AE,Ld,0.4,0.4,10,0.231699,0.008275,0.632734,...,0.019394,0.001052,191.8,9.631199,1.0,0.0,13.0,1.000000,0.258093,0.020927
96,LocPop,Random-25,AE,S,0.4,0.4,10,0.212767,0.013157,0.603318,...,0.039497,0.002175,313.0,17.372392,1.0,0.0,12.9,0.700000,0.000000,0.000000


## Verify convergence and output coverage

In [3]:
heuristic = summary[summary["Method"] == "LocPop"]
print("Summary rows:", len(summary))
print("Datasets:", sorted(summary["Dataset"].unique()))
print("Minimum convergence rate:", heuristic["Converged"].min())
print("Maximum recorded moves:", heuristic["Moves"].max())

if not (heuristic["Converged"] == 1.0).all():
    display(heuristic[heuristic["Converged"] < 1.0])
    raise RuntimeError("At least one run reached the move cap; do not use the outputs without investigation.")

display(summary.sort_values(["Dataset", "Method", "Initialization", "Preference"]).head(20))
print("Artifacts written under csv/PopularCommunity and figures/PopularCommunity")


Summary rows: 98
Datasets: ['Cora', 'Jazz', 'Karate Club', 'Random-25']
Minimum convergence rate: 1.0
Maximum recorded moves: 3550.6


,Method,Dataset,Preference,Initialization,Beta Friend,Beta Enemy,Repetitions,Adjusted Rand Index,Adjusted Rand Index SD,Modularity,...,Seconds,Seconds SD,Moves,Moves SD,Converged,Converged SD,Final Coalitions,Final Coalitions SD,Initial Adjusted Rand Index,Initial Adjusted Rand Index SD
30,Leiden,Cora,-,-,NaN,NaN,10,0.143650,0.013804,0.798567,...,3.294000,0.186878,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,LocPop,Cora,AE,Ld,0.20,0.20,10,0.163550,0.001147,0.635644,...,0.924752,0.071152,1438.4,47.113056,1.0,0.0,127.4,2.690725,3.757192e-01,0.027512
41,LocPop,Cora,AE,Ld,0.25,0.35,10,0.037554,0.000731,0.381460,...,0.916903,0.050078,1623.0,48.664155,1.0,0.0,127.4,2.690725,7.680006e-02,0.008798
47,LocPop,Cora,AE,Ld,0.40,0.40,10,-0.001569,0.000018,0.145750,...,1.171856,0.049641,2078.7,31.394426,1.0,0.0,118.0,0.894427,2.082574e-02,0.002400
33,LocPop,Cora,AF,Ld,0.20,0.20,10,0.164417,0.001374,0.638039,...,0.898049,0.062469,1381.0,30.272099,1.0,0.0,127.4,2.690725,3.779028e-01,0.028193
39,LocPop,Cora,AF,Ld,0.25,0.35,10,0.037554,0.000279,0.379274,...,0.909690,0.056505,1607.6,52.896503,1.0,0.0,127.4,2.690725,7.655384e-02,0.008745
45,LocPop,Cora,AF,Ld,0.40,0.40,10,-0.001489,0.000008,0.145479,...,13.461619,36.889229,2081.5,30.981446,1.0,0.0,116.8,0.400000,2.082023e-02,0.002402
31,LocPop,Cora,B,Ld,0.20,0.20,10,0.163914,0.001471,0.637506,...,0.895823,0.057392,1393.4,30.020660,1.0,0.0,127.4,2.690725,3.770794e-01,0.027852
37,LocPop,Cora,B,Ld,0.25,0.35,10,0.037371,0.000264,0.381144,...,0.914500,0.063390,1608.4,48.957533,1.0,0.0,127.4,2.690725,7.667137e-02,0.008773
43,LocPop,Cora,B,Ld,0.40,0.40,10,-0.001540,0.000007,0.145597,...,1.187368,0.081963,2079.0,30.977411,1.0,0.0,118.0,0.447214,2.082321e-02,0.002402


Artifacts written under csv/PopularCommunity and figures/PopularCommunity
